In [ ]:
# -- Cell 1 -- rclone + Drive. Same pattern as the previous three notebooks.# Requires: Settings -> Internet ON, Accelerator GPU T4 x2, RCLONE_DRIVE_TOKEN# attached to THIS notebook.import os, subprocessr = subprocess.run("curl -s https://rclone.org/install.sh | sudo bash", shell=True)if r.returncode not in (0, 3):    raise RuntimeError("rclone install failed (exit %d)" % r.returncode)from kaggle_secrets import UserSecretsClienttoken = UserSecretsClient().get_secret("RCLONE_DRIVE_TOKEN")os.makedirs("/root/.config/rclone", exist_ok=True)with open("/root/.config/rclone/rclone.conf", "w") as f:    f.write("[drive]\ntype = drive\nscope = drive\ntoken = " + token + "\n")REMOTE = "drive:Distillation"out = subprocess.run("rclone lsf " + REMOTE, shell=True, capture_output=True, text=True)print(out.stdout or out.stderr)assert out.returncode == 0, "cannot see " + REMOTE

In [ ]:
# -- Cell 2 -- deps + GPUs.subprocess.run('pip install -q -U "transformers>=5.0" pyarrow', shell=True, check=True)subprocess.run("pip uninstall -y -q torchao", shell=True)   # peft/torchao clash guardimport torch, numpy as np, pandas as pd, glob, json, timeprint("torch", torch.__version__, "| GPUs", torch.cuda.device_count())for i in range(torch.cuda.device_count()):    p = torch.cuda.get_device_properties(i)    print("   cuda:%d %s %.0f GB" % (i, p.name, p.total_memory / 1e9))assert torch.cuda.device_count() >= 1, "need a GPU"if torch.cuda.device_count() < 2:    print("\nWARNING: only one GPU. The two arms will run SEQUENTIALLY, roughly "          "doubling wall clock. Set Accelerator to T4 x2 for the parallel path.")

In [ ]:
# -- Cell 3 -- pull the four inputs.##   distill/       our training code            (small)#   subset parquet 2M molecules + descriptors   (~0.6 GB)#   teacher cache  41 shards, both masks        (~15 GB)  <- the big one#   student init   their released 32M MLM       (~0.13 GB)## The cache is the only slow item. It is pulled once per session; there is no way# around that short of making it a Kaggle Dataset, which is worth doing if you end# up running this more than twice.WORK = "/kaggle/working"CODE, SUBSET_D, CACHE, INIT = (WORK + "/distill", WORK + "/subset",                               "/tmp/cache", WORK + "/init")def pull(remote, local, extra=""):    os.makedirs(local, exist_ok=True)    subprocess.run("rclone copy %s/%s %s --transfers 8 %s -P" % (REMOTE, remote, local, extra),                   shell=True, check=True)if not os.path.exists(CODE + "/train.py"):    pull("distill", CODE)if not glob.glob(SUBSET_D + "/*.parquet"):    pull("data/pretrain_subset_2M", SUBSET_D, "--include '*.parquet'")if not os.path.exists(INIT + "/model.safetensors"):    pull("models/peptideclm-2-mlm-small", INIT)# The cache goes to /tmp: 15 GB would crowd the 20 GB /kaggle/working quota# alongside checkpoints. Same lesson as the CellPPD checkpoints.n_have = len(glob.glob(CACHE + "/*.npz"))if n_have < 41:    print("pulling teacher cache (~15 GB, several minutes)...")    pull("data/teacher_cache_2M", CACHE, "--include '*.npz'")SUBSET = glob.glob(SUBSET_D + "/*.parquet")[0]shards = sorted(glob.glob(CACHE + "/*.npz"))print("\ncode    :", sorted(os.path.basename(f) for f in glob.glob(CODE + "/*.py")))print("subset  : %s  %.2f GB" % (SUBSET, os.path.getsize(SUBSET) / 1e9))print("init    : %.2f GB" % (os.path.getsize(INIT + "/model.safetensors") / 1e9))print("cache   : %d shards  %.2f GB"      % (len(shards), sum(os.path.getsize(f) for f in shards) / 1e9))assert len(shards) == 41, "expected 41 shards, got %d -- cache is incomplete" % len(shards)

In [ ]:
# -- Cell 4 -- verify the inputs agree with each other before spending hours.## The failure this catches: a cache built against a DIFFERENT subset than the one# loaded here. Nothing would crash -- the descriptors and teacher targets would# simply belong to different molecules, and the run would train on noise.import syssys.path.insert(0, CODE)from transformers import AutoTokenizermeta = pd.read_parquet(SUBSET, columns=["source", "smiles", "n_tokens"])tok = AutoTokenizer.from_pretrained(INIT, trust_remote_code=True)idx, npos = [], 0for f in shards:    with np.load(f) as z:        idx.append(z["mol_idx"]); npos += int(z["ptr_a"][-1])idx = np.concatenate(idx)print("subset molecules {:,} | cache molecules {:,}".format(len(meta), len(idx)))assert len(idx) == len(meta) and idx.min() == 0 and idx.max() == len(meta) - 1, \    "cache does not cover the subset one-to-one"assert len(np.unique(idx)) == len(idx), "cache has duplicate molecules"# Spot-check alignment: re-tokenize a molecule and confirm every cached masked# position falls inside it. A mismatched subset shows up here immediately.with np.load(shards[0]) as z:    bad = 0    for r in range(200):        m = int(z["mol_idx"][r])        L = len(tok(meta.smiles.iloc[m], add_special_tokens=False)["input_ids"])        p = z["pos_a"][z["ptr_a"][r]:z["ptr_a"][r + 1]]        if len(p) and (p.min() < 1 or p.max() > L):            bad += 1print("alignment spot-check: %d/200 molecules out of range" % bad)assert bad == 0, "cache positions do not match this subset's tokenization"print("masking rate %.4f" % (npos / (meta.n_tokens.sum() - 2 * len(meta))))print("\ninputs are consistent")

In [ ]:
# -- Cell 5 -- unit tests, then a short real-data smoke run on ONE shard.# ~4 minutes against ~8 hours. Everything below has already been verified on an# RTX 3050; this re-checks it on the actual T4s and the actual pulled data.r = subprocess.run(["python", "test_smoke.py", "--subset", SUBSET, "--cache", CACHE,                    "--student", INIT, "--device", "cuda", "--max-tokens", "8192"],                   cwd=CODE, capture_output=True, text=True)print(r.stdout[-3500:])if r.returncode != 0:    print("----- STDERR -----"); print(r.stderr[-3000:])assert r.returncode == 0, "smoke tests failed -- do not start the real run"

In [ ]:
# -- Cell 6 -- launch both arms, one per GPU.##   GPU 0  treatment   KD + MTR + SPKD        (teacher signal)#   GPU 1  control     hard-label MLM + MTR   (no teacher)## Same init, same data, same schedule, same step count. The ONLY difference is# whether the teacher is in the loss. Without the control arm, "our student beats# their released 32M" is unfalsifiable -- the gain could just be 640M extra tokens# of training. Running them concurrently costs nothing extra in wall clock.## Both resume from their own latest.pt, so re-running this cell after a dead# session continues rather than restarting.OUT = WORK + "/runs"DEST = REMOTE + "/results/distill"os.makedirs(OUT, exist_ok=True)print("pulling any previous run state to resume...")subprocess.run("rclone copy %s %s --transfers 8 -P" % (DEST, OUT), shell=True, check=False)ARMS = [("treatment", 0), ("control", 1)]if torch.cuda.device_count() < 2:    ARMS = [("treatment", 0), ("control", 0)]      # sequential fallbackprocs = []for arm, gpu in ARMS:    d = "%s/%s" % (OUT, arm)    os.makedirs(d, exist_ok=True)    log = open("/tmp/%s.log" % arm, "w")    cmd = ["python", "-u", "train.py", "--arm", arm, "--subset", SUBSET,           "--cache", CACHE, "--init", INIT, "--out", d,           "--epochs", "2", "--max-tokens", "16384",           "--calibrate-at", "2500", "--log-every", "200", "--save-every", "500"]    p = subprocess.Popen(cmd, cwd=CODE, stdout=log, stderr=subprocess.STDOUT,                         env=dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu)))    procs.append((arm, p))    print("launched %-10s on GPU %d (pid %d)" % (arm, gpu, p.pid))    if len(ARMS) == 2 and torch.cuda.device_count() < 2:        p.wait()                                   # sequential fallback# Mirror to Drive every 10 min. Checkpoints are ~380 MB each (32M params plus# AdamW state), so this is cheap, and a session killed at hour 7 keeps its work.SYNC = subprocess.Popen(    "while true; do rclone copy %s %s --drive-chunk-size 64M "    ">> /tmp/rclone_sync.log 2>&1; sleep 600; done" % (OUT, DEST), shell=True)print("background sync -> Drive every 10 min\n")def tail(arm, n=1):    try:        L = [l.rstrip() for l in open("/tmp/%s.log" % arm) if l.startswith("[" + arm)]        return L[-n] if L else "(starting)"    except OSError:        return "(no log)"t0 = time.time()while any(p.poll() is None for _, p in procs):    time.sleep(300)    print("[%6.1f min]" % ((time.time() - t0) / 60))    for arm, p in procs:        print("   %-10s %s | %s" % (arm, "running" if p.poll() is None else                                    "exit %d" % p.returncode, tail(arm)))SYNC.terminate()subprocess.run("pkill -f 'rclone copy %s' || true" % OUT, shell=True)for arm, p in procs:    print("%s exit %d" % (arm, p.returncode))    if p.returncode != 0:        print(open("/tmp/%s.log" % arm).read()[-3000:])print("\nelapsed %.1f min" % ((time.time() - t0) / 60))

In [ ]:
# -- Cell 7 -- compare the two arms, then ship everything.## This is only a TRAINING-loss comparison. It says whether the teacher signal# changed the optimisation, not whether the student is better at anything anyone# cares about. The real verdict comes from finetuning both students on CellPPD /# AmpHGT / THPep against the baselines already reproduced -- a separate notebook.import matplotlib.pyplot as pltH = {}for arm, _ in ARMS:    f = "%s/%s/history.json" % (OUT, arm)    if os.path.exists(f):        H[arm] = pd.DataFrame(json.load(open(f)))        print("%-10s %d steps | final loss %.4f" % (arm, len(H[arm]), H[arm].loss.iloc[-1]))if H:    fig, ax = plt.subplots(1, 3, figsize=(15, 4))    for arm, h in H.items():        w = h.rolling(200, min_periods=1).mean()        ax[0].plot(h.step, w.loss, label=arm)        ax[1].plot(h.step, w.mtr, label=arm)        if "kd" in h:            ax[2].plot(h.step, w.kd, label="treatment KD")        if "mlm" in h:            ax[2].plot(h.step, w.mlm, label="control MLM")    for i, t in enumerate(["total (not comparable across arms)", "MTR (same term both arms)",                           "KD vs MLM"]):        ax[i].set_title(t); ax[i].set_xlabel("step"); ax[i].legend(); ax[i].grid(alpha=.3)    plt.tight_layout(); plt.savefig(OUT + "/curves.png", dpi=120); plt.show()    # MTR is the one term defined identically in both arms, so it is the only    # honest head-to-head here: it asks whether the teacher signal helped the    # student learn physicochemistry better than hard labels did.    print("\nfinal MTR (identical term, lower is better):")    for arm, h in H.items():        print("   %-10s %.5f" % (arm, h.mtr.tail(500).mean()))subprocess.run("rclone copy %s %s --drive-chunk-size 64M -P" % (OUT, DEST),               shell=True, check=True)print("\nuploaded to " + DEST)print(subprocess.run("rclone lsf -R " + DEST, shell=True,                     capture_output=True, text=True).stdout)